# 04 — VRAM Batch-Size Finder (real probe)

**This replaces the previous, broken "finder"** which never measured anything — it merely
re-printed a stale cached value (`8`, calibrated for the much larger YOLO26**l**). For
YOLO26**n** at 1280×1280 that value is far too small and wastes the GPU.

This notebook performs a **genuine** search for the largest batch size that trains
without running out of memory, then keeps a small safety margin:

1. For each candidate batch size we launch `scripts/batch_finder.py` as an **isolated
   subprocess** that runs one real 1-epoch training pass (full Ultralytics pipeline:
   model + AMP + optimizer + augmentation) on a tiny probe subset and reports peak
   reserved VRAM. Isolation guarantees an OOM at one size cannot corrupt the next
   measurement via leaked/fragmented memory.
2. We probe a **coarse** grid first, then **refine** between the largest size that fit
   and the first that OOM’d, stepping by 2, to pin the exact maximum.
3. The chosen batch is the largest size that passed the real probe (optionally minus a
   small safety step for long unattended runs). It is written to
   `runs/optimal_batch.json` and consumed by `05_train_yolo26n.ipynb`.

Configuration is **per run** (model variant × imgsz × GPU). Re-run this whenever any of
those change.

In [ ]:
import os, json, subprocess, sys
from pathlib import Path

ROOT = Path('/home/jovyan/shared/s0598584')
WEIGHTS = str(ROOT/'yolo26n.pt')          # YOLO26n pretrained init
IMGSZ   = 1280                            # tile size = imgsz (no resize)
DATA    = str(ROOT/'dataset_face_lp'/'dataset.yaml')   # nc=2 face+license-plate dataset
PROBE_SCRIPT = str(ROOT/'scripts'/'batch_finder.py')
SAFETY_STEP  = 0                          # subtract this many from the max that fit (0 = use the exact max)

for p in (WEIGHTS, DATA, PROBE_SCRIPT):
    assert os.path.exists(p), f'missing: {p}'
import torch
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
print('weights   :', WEIGHTS)
print('imgsz     :', IMGSZ)
print('data      :', DATA)
print('GPU       :', torch.cuda.get_device_name(0), f'({TOTAL_VRAM:.1f} GB)')

In [ ]:
# Build a small probe subset (>= largest candidate batch) from the nc=2 train split.
# Memory depends on batch x imgsz x model, not on image content, so a small subset suffices.
import shutil, random
SRC = ROOT/'dataset_face_lp'
P   = ROOT/'dataset_probe'
if P.exists():
    shutil.rmtree(P)
for s in ['train','val']:
    (P/'images'/s).mkdir(parents=True); (P/'labels'/s).mkdir(parents=True)
pos = sorted(f for f in os.listdir(SRC/'labels'/'train')
             if os.path.getsize(SRC/'labels'/'train'/f) > 0)[:400]
for fn in pos:
    stem = os.path.splitext(fn)[0]
    for s in ['train','val']:
        shutil.copy(SRC/'labels'/'train'/fn, P/'labels'/s/fn)
        os.symlink(os.path.realpath(SRC/'images'/'train'/(stem+'.png')), P/'images'/s/(stem+'.png'))
(P/'dataset.yaml').write_text(
    f'path: {P}\ntrain: images/train\nval: images/val\ntest: images/val\nnc: 2\nnames:\n  0: face\n  1: license-plate\n')
PROBE_DATA = str(P/'dataset.yaml')
print('probe subset:', len(pos), 'tiles ->', PROBE_DATA)

In [ ]:
# Probe one batch size in a clean subprocess. Returns (fit: bool, peak_gb: float|None).
def probe(bs):
    r = subprocess.run([sys.executable, PROBE_SCRIPT, str(bs), str(IMGSZ), WEIGHTS, PROBE_DATA],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    peak = None
    for line in r.stdout.splitlines():
        if line.startswith('PROBE_OK'):
            peak = json.loads(line[len('PROBE_OK'):]).get('peak_gb')
    fit = (r.returncode == 0 and peak is not None)
    print(f'  batch={bs:>3}  ->  {"OK " if fit else "OOM"}  peak={peak if peak else "-":>6} GB')
    if not fit and 'PROBE_OOM' not in r.stdout:
        print('   (last lines)\n   ' + '\n   '.join(r.stdout.strip().splitlines()[-4:]))
    return fit, peak

results = {}
print('Coarse grid:')
for bs in [8, 16, 24, 32, 40, 48]:
    fit, peak = probe(bs); results[bs] = (fit, peak)
    if not fit:
        break

In [ ]:
# Refine between the largest size that fit and the first that OOM'd (step of 2).
passed = sorted(b for b, (f, _) in results.items() if f)
failed = sorted(b for b, (f, _) in results.items() if not f)
lo = max(passed)
hi = min(failed) if failed else None
if hi is not None and hi - lo > 2:
    print(f'Refining between {lo} (OK) and {hi} (OOM):')
    for bs in range(lo + 2, hi, 2):
        fit, peak = probe(bs); results[bs] = (fit, peak)
        if not fit:
            break
print('\nAll probes:', {b: ('OK' if f else 'OOM', p) for b, (f, p) in sorted(results.items())})

In [ ]:
# Pick the largest batch that actually trained (the real-train probe already validates it),
# minus an optional safety step, and persist it for the training notebook.
max_fit = max(b for b, (f, _) in results.items() if f)
optimal = max(8, max_fit - SAFETY_STEP)
peak_at_optimal = results.get(optimal, (None, None))[1] or results[max_fit][1]

payload = {
    'optimal_batch': int(optimal),
    'max_fit_batch': int(max_fit),
    'imgsz': IMGSZ,
    'model': 'yolo26n',
    'peak_vram_gb': peak_at_optimal,
    'total_vram_gb': round(TOTAL_VRAM, 2),
    'safety_step': SAFETY_STEP,
    'probes': {str(b): {'fit': f, 'peak_gb': p} for b, (f, p) in sorted(results.items())},
}
out = ROOT/'runs'/'optimal_batch.json'
out.write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print('\nOPTIMAL_BATCH =', optimal, '(max that fit:', max_fit, f'@ {peak_at_optimal} GB of {TOTAL_VRAM:.1f} GB)')
print('written ->', out)

# clean up probe subset
shutil.rmtree(P, ignore_errors=True)

## ✅ Phase 4 done

`runs/optimal_batch.json` now holds a **measured** optimal batch size for YOLO26n @ 1280
on this GPU (plus the full probe trace for transparency). Continue with
`05_train_yolo26n.ipynb`.